# 03 — Portfolio Optimization, Efficient Frontier & Monte Carlo

Demonstrates `src/portfolio/` (returns, risk, optimization, efficient_frontier) and `src/visualization/charts.py`.

- Returns computed from prices; annualization assumes 252 trading days.
- Optimized portfolios solve `min w'Σw` or `max Sharpe` via SLSQP with `Σw=1` and optional `max_weight`.
- The frontier sweeps target returns; Monte Carlo draws 5k random allocations with fixed seed `42`.

In [ ]:
import pandas as pd, numpy as np
from src.portfolio.returns import simple_returns, annualized_return, annualized_volatility
from src.portfolio.risk import sharpe_ratio, max_drawdown, correlation_matrix, covariance_matrix
from src.portfolio.optimization import minimum_variance_portfolio, maximum_sharpe_portfolio, target_return_portfolio, monte_carlo_simulation
from src.portfolio.efficient_frontier import efficient_frontier
from src.visualization.charts import plot_cumulative_returns, plot_efficient_frontier, plot_correlation_heatmap, plot_drawdown

np.random.seed(42)
dates = pd.date_range("2018-01-01", periods=1260, freq="B")
tickers = ["AAPL","MSFT","JPM","XOM","TLT"]
mus  = np.array([0.0006,0.0005,0.0003,0.0002,0.0001])
vols = np.array([0.015,0.013,0.014,0.012,0.006])
prices = pd.DataFrame(index=dates)
for t, mu, vol in zip(tickers, mus, vols):
    prices[t] = 100 * np.cumprod(1 + np.random.normal(mu, vol, len(dates)))
print("Synthetic demonstration dataset — not actual company financials.")
prices.head()


## 1. Returns & risk (explicit annualization)
`annualized_return = Π(1+r)^{252/N} -1`, `ann_vol = σ_daily·√252`, Sharpe with `Rf` per-period.

In [ ]:
rets = simple_returns(prices)
rets.describe().round(4)


In [ ]:
ann_ret = annualized_return(rets)
ann_vol = annualized_volatility(rets)
pd.DataFrame({"Ann Return": ann_ret, "Vol": ann_vol, "Sharpe": sharpe_ratio(rets, risk_free_rate=0.02)}).round(4)


In [ ]:
correlation_matrix(rets).round(2)


In [ ]:
print("Max DD AAPL:", f"{max_drawdown(prices=prices['AAPL']):.1%}")
plot_cumulative_returns(prices=prices, save_path="../outputs/cumulative_returns.png")
plot_correlation_heatmap(correlation_matrix(rets), save_path="../outputs/correlation.png")


## 2. Optimization
All three use `scipy.optimize.minimize` (SLSQP) with `Σw=1`. Try invalid `max_weight=0.1` for 5 assets — it raises `ValueError: n*max_weight <1`.

In [ ]:
er  = annualized_return(rets)
cov = covariance_matrix(rets)
min_var = minimum_variance_portfolio(er, cov, risk_free_rate=0.02)
max_sh  = maximum_sharpe_portfolio(er, cov, risk_free_rate=0.02)
target  = target_return_portfolio(er, cov, target_return=0.10, risk_free_rate=0.02)
for name, res in [("Min Var",min_var),("Max Sharpe",max_sh),("Target 10%",target)]:
    print(f"\n{name}: Ret={res['expected_return']:.2%} Vol={res['volatility']:.2%} Sharpe={res['sharpe_ratio']:.2f}")
    print(res["weights"].round(3).to_dict())


## 3. Efficient frontier
Sweeps target returns from min-var return to `max(μ)`; each point is a constrained `min_vol` solution.

In [ ]:
frontier = efficient_frontier(er, cov, n_points=30, risk_free_rate=0.02)
frontier.head().round(4)


In [ ]:
mc = monte_carlo_simulation(er, cov, n_portfolios=3000, risk_free_rate=0.02, seed=42)
mc.head()


In [ ]:
plot_efficient_frontier(frontier, min_var=min_var, max_sharpe=max_sh, mc_df=mc, save_path="../outputs/efficient_frontier.png")


## 4. Drawdown of optimized portfolio
Reconstruct weighted returns to show path-dependency that Sharpe alone hides.

In [ ]:
weights = max_sh["weights"]
port_rets = (rets * weights).sum(axis=1)
plot_drawdown(returns=port_rets, title="Max-Sharpe Drawdown", save_path="../outputs/drawdown.png")


## 5. Export
```python
pd.DataFrame({"MinVar":min_var['weights'], "MaxSharpe":max_sh['weights']}).to_csv("outputs/optimal_portfolios.csv")
frontier.to_csv("outputs/efficient_frontier.csv")
```
See `python -m src.main --demo` for the full exported set.